# BraTS 2024 - Slice Dataset Generation and Export

This notebook handles:
1. Environment configuration, path resolution, and `brain_mri` imports.
2. (Optional) Unpacking the `BraTS2024-BraTS-GLI-TrainingData.zip` archive.
3. Loading preprocessing configuration and patient splits (`train` / `validate` / `test`).
4. **Single-patient sanity check:** verifying slice extraction, PyTorch tensors, and visualization.
5. **Full dataset export:** saving 2D slices as `.pt` files to disk with automatic resumption.

## 1. Environment Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Resolve repository root directory
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

repo_src = PROJECT_ROOT / "src"
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

print(f"Project root: {PROJECT_ROOT}")

import torch
import matplotlib.pyplot as plt

from brain_mri import load_data_config
from brain_mri.data.split import get_or_create_splits
from brain_mri.data.slice_extraction import extract_patient_slices
from brain_mri.data.dataset_export import save_patients_to_pt, export_dataset_split

## 2. Data Paths Configuration and ZIP Extraction

In [ ]:
# Data directory and ZIP archive path
DATA_DIR = Path(r"E:\abronikowski\data")
ZIP_PATH = DATA_DIR / "BraTS2024-BraTS-GLI-TrainingData.zip"

# Directory for extracted raw data
RAW_DATA_DIR = DATA_DIR / "raw"

# Directory containing individual patient folders (ZIP extracts into 'training_data1_v2')
GLI_ROOT = RAW_DATA_DIR / "training_data1_v2"
if not GLI_ROOT.exists() and RAW_DATA_DIR.exists():
    if (RAW_DATA_DIR / "BraTS2024-BraTS-GLI-TrainingData").exists():
        GLI_ROOT = RAW_DATA_DIR / "BraTS2024-BraTS-GLI-TrainingData"
    else:
        for cand in [RAW_DATA_DIR] + [p for p in RAW_DATA_DIR.iterdir() if p.is_dir()]:
            if any(p.name.startswith("BraTS") for p in cand.iterdir()):
                GLI_ROOT = cand
                break

# Output directory for exported 2D slice tensors (.pt files)
PROCESSED_ROOT = DATA_DIR / "processed"

print(f"ZIP path:       {ZIP_PATH} (exists: {ZIP_PATH.exists()})")
print(f"GLI root:       {GLI_ROOT} (exists: {GLI_ROOT.exists()})")
print(f"Processed root: {PROCESSED_ROOT}")

In [ ]:
# Optional: unpack ZIP directly from Python (run only if raw data is not yet extracted)
import zipfile
from tqdm import tqdm

if ZIP_PATH.exists() and not GLI_ROOT.exists():
    print(f"Extracting {ZIP_PATH} to {RAW_DATA_DIR}...")
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        members = zip_ref.infolist()
        for member in tqdm(members, desc="Unpacking ZIP"):
            zip_ref.extract(member, RAW_DATA_DIR)
    print("Extraction complete.")
    # Refresh GLI_ROOT
    if (RAW_DATA_DIR / "training_data1_v2").exists():
        GLI_ROOT = RAW_DATA_DIR / "training_data1_v2"
else:
    print("Target data directory exists or ZIP not found; skipping extraction.")

## 3. Preprocessing Configuration and Splits (`train` / `val` / `test`)

In [ ]:
# Load preprocessing YAML configuration
config = load_data_config(PROJECT_ROOT / "configs" / "data_config.yaml")
config.paths.z_boundaries = PROJECT_ROOT / "data" / "patients_z_boundaries.json"

# Load or generate patient splits (70% train / 15% validation / 15% test)
splits_json_path = DATA_DIR / "data_splits.json"
splits = get_or_create_splits(folder_path=GLI_ROOT, json_path=splits_json_path)

print("Patient count by split:")
print(f" - Train:      {len(splits['train'])}")
print(f" - Validation: {len(splits['validate'])}")
print(f" - Test:       {len(splits['test'])}")

## 4. Single-Patient Sanity Check and Visualization

In [ ]:
# Test slice extraction with the first training patient
test_patient_id = splits["train"][0]
test_patient_dir = GLI_ROOT / test_patient_id
print(f"Testing patient: {test_patient_id}")

# Extract slices
slices = extract_patient_slices(test_patient_dir, config=config)
print(f"Extracted {len(slices)} slices.")

# Save slices to temporary directory for verification
test_output_dir = DATA_DIR / "test_processed"
saved_files = save_patients_to_pt(slices, test_output_dir)
print(f"Saved {len(saved_files)} .pt files to {test_output_dir}")

# Inspect a middle slice (where tumor is typically visible)
sample_file = saved_files[len(saved_files) // 2]
data = torch.load(sample_file, weights_only=True)

img = data["image"]  # (4, 224, 224)
mask = data["mask"]  # (224, 224)

print("\n--- Tensor Verification ---")
print(f"Image shape:        {img.shape}, dtype: {img.dtype}")
print(f"Mask shape:         {mask.shape}, dtype: {mask.dtype}")
print(f"Unique mask labels: {torch.unique(mask).tolist()}")
print(f"Background value:   {img[:, 0, 0].tolist()}")

# Plot all 4 MRI modalities + segmentation mask
modalities = ["T1", "T1CE", "T2", "FLAIR"]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i in range(4):
    axes[i].imshow(img[i].numpy(), cmap="gray")
    axes[i].set_title(modalities[i])
    axes[i].axis("off")

axes[4].imshow(mask.numpy(), cmap="tab10", vmin=0, vmax=4)
axes[4].set_title(f"Mask (z={data['z_idx']})")
axes[4].axis("off")

plt.tight_layout()
plt.show()

## 5. Full Dataset Export

`export_dataset_split` supports automatic resumption: patients that were already completely exported in previous runs will be skipped automatically.

In [ ]:
stats = export_dataset_split(
    data_root=GLI_ROOT,
    splits=splits,
    output_root=PROCESSED_ROOT,
    config=config,
)
print("Export summary:", stats)

## 6. Verify Exported Slices

In [ ]:
for split_name in ["train", "validate", "test"]:
    split_folder = PROCESSED_ROOT / split_name
    if split_folder.exists():
        count = len(list(split_folder.rglob("*.pt")))
        print(f"Split '{split_name}': {count} .pt files")
    else:
        print(f"Split '{split_name}': directory does not exist")